# Example 0.13.3 / 5: Exercise 19-4: Choosing a Standard as a Bimatrix Game

Companion notebook for **0.13.3 Exercises on Special Structures** in the AMPL lab Documentation, section "Exercise 19-4: Choosing a Standard as a Bimatrix Game". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Exercise 19-4: Choosing a Standard as a Bimatrix Game

Two companies each adopt standard 1 or 2. The payoffs are $A = \begin{pmatrix}10 & 3\\ 2 & 9\end{pmatrix}$
for company A and $B = \begin{pmatrix}4 & 6\\ 7 & 5\end{pmatrix}$ for company B, with A's choice
indexing rows. Exercise 19-3 shows that a Nash equilibrium satisfies complementarity conditions:
each probability is nonnegative, each player's expected return from a play is at most that
player's best expected return, and a play is used only when it attains that best return
(Complementarity Constraints (AMPL lab documentation, section 11.6), Equilibrium Problems and MPCC (AMPL lab documentation, section 11.6.1)).

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param nA > 0;                               # plays available to A
param nB > 0;                               # plays available to B
param payoffA {1..nA, 1..nB};               # A's winnings
param payoffB {1..nA, 1..nB};               # B's winnings

var PlayA {1..nA};                          # A's mixed strategy
var PlayB {1..nB};                          # B's mixed strategy
var MaxExpA;                                # A's best expected return
var MaxExpB;                                # B's best expected return

# a play is used only if its expected return is the best one
subject to ComplA {i in 1..nA}:
    0 <= PlayA[i] complements MaxExpA - sum {j in 1..nB} payoffA[i,j] * PlayB[j] >= 0;
subject to ComplB {j in 1..nB}:
    0 <= PlayB[j] complements MaxExpB - sum {i in 1..nA} payoffB[i,j] * PlayA[i] >= 0;
# probabilities sum to 1; the best returns are the free variables of these equations
subject to TotalA: sum {i in 1..nA} PlayA[i] = 1;
subject to TotalB: sum {j in 1..nB} PlayB[j] = 1;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["nA"] = 2
ampl.param["nB"] = 2
ampl.param["payoffA"] = {(1, 1): 10, (1, 2): 3, (2, 1): 2, (2, 2): 9}
ampl.param["payoffB"] = {(1, 1): 4, (1, 2): 6, (2, 1): 7, (2, 2): 5}

In [5]:
# run.py
ampl.solve(solver="path", verbose=False)  # hide PATH's iteration log
print(ampl.solve_result)
print("A:", {i: round(p, 4) for i, p in ampl.var["PlayA"].to_dict().items()})
print("B:", {j: round(p, 4) for j, p in ampl.var["PlayB"].to_dict().items()})

solved
A: {1: 0.5, 2: 0.5}
B: {1: 0.4286, 2: 0.5714}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param nA := 2;
param nB := 2;
param payoffA:  1   2 :=
1              10   3
2               2   9;
param payoffB:  1   2 :=
1               4   6
2               7   5;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver path;
solve > path.log;      # send PATH's iteration log to a file
remove path.log;       # and delete it
display solve_result, PlayA, PlayB;

Path 5.0.05: Solution found.
3 iterations (0 for crash); 3 pivots.
16 function, 4 gradient evaluations.
solve_result = solved

: PlayA    PlayB      :=
1   0.5   0.428571
2   0.5   0.571428
;

